# 第4回：モデルを比較し、特徴量と実験で改善する

この回は3つのパートで構成します：**複数のモデルを同じ条件で比較する ／ 化学の知識を特徴量にする ／ 改善実験を1つずつ行う**。

**セルの動かし方**：各セル（灰色の枠）を選んで `Shift + Enter`（またはセル左の▷ボタン）を押すと実行できます。
**上から順に**実行してください。前のセルを飛ばすと、後のセルでエラーになります。

**AIと一緒に進める**：分からないコードは、セル全体ではなく気になる数行をM365 CopilotなどのAIへ貼り、
説明や修正を相談します。ただし、提案されたコードは必ず実行結果を見て確かめます。

まず「基本」と「演習」を進めます。「補足」は必要に応じて読み、
「発展（任意）」「追加演習（任意）」「自由課題（任意）」は飛ばしても構いません。


In [ ]:
# 【準備セル】教材フォルダの場所を自動で見つけます。中身は今は理解しなくてOK、そのまま実行してください。
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回で扱うこと

複数モデルを公平に比べ、知識を特徴量に変え、1要素ずつ条件を変えて改善の効果を検証します。

### 進め方

この回は3つのパートに分かれています。パート1から順に「基本」と「演習」を進めてください。
1日で終える必要はありません。「発展（任意）」と「追加演習（任意）」は、余裕がある場合だけ取り組みます。

### 用語について

初めて出る用語は、その用語を使うセルで説明します。ここでまとめて暗記する必要はありません。

> **実行前の30秒予想**：各パートの問いに、今の言葉で仮の答えを書いてから始めます。


---

# パート1：複数のモデルを同じ条件で比較する

**このパートの問い：複雑なモデルは本当にいつも優れているか。**


## 「複雑なモデルほど強い」は本当か

新しいモデルを次々試したくなりますが、この回で確かめるのは**「複雑さは必ずしも勝たない」**という
実感です。大事なのは勝ち負けそのものより、**フェアな比べ方**を身につけること。フェアな比較には
3つの「同じ」が要ります：**同じ分割・同じ指標・同じ前処理**。

まず、単純〜複雑まで5つのモデルを1つの辞書にまとめます。前処理が要るモデルは`make_pipeline`で
前処理込みにしてあるので、どれも同じ`X_train`をそのまま渡せます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
import time
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import f1_score

features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa", "h_bond_donors", "rotatable_bonds"]
X_train, X_valid, y_train, y_valid = train_test_split(df[features], df["active"], test_size=0.25, random_state=42, stratify=df["active"])
models = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "Logistic": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000)),
    "Tree": make_pipeline(SimpleImputer(strategy="median"), DecisionTreeClassifier(max_depth=4, random_state=42)),
    "Random Forest": make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)),
    "Gradient Boosting": HistGradientBoostingClassifier(max_iter=200, random_state=42),
}


## 演習：同じ土俵で、F1と学習時間を並べる

5モデルを同じデータで学習し、**検証F1**と**学習にかかった秒数**を並べます。性能だけでなく
**コスト（時間）**も一緒に見るのが実務的な比較です。


In [ ]:
rows = []
for name, model in models.items():
    start = time.perf_counter()
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - start
    rows.append({"モデル": name, "検証F1": f1_score(y_valid, model.predict(X_valid)), "学習秒": elapsed})
pd.DataFrame(rows).sort_values("検証F1", ascending=False).round({"検証F1": 3, "学習秒": 4})


### 出力の読み方

- **Dummyが最下位**なのは当然。他がDummyをどれだけ引き離すかが価値です。
- **最も複雑なモデルが1位とは限りません**。線形モデルが健闘したり、木モデルと僅差だったりします。差が小さいなら、**速くて説明しやすいモデル**を選ぶ理由になります。
- ただし、これは**1回の分割の結果**。順位が分割運で入れ替わるかもしれません。次で交差検証により安定性を確かめます。


## 補足：交差検証で「安定して強いか」を見る

1回の勝敗は運に左右されます。交差検証で**平均F1・ばらつき(標準偏差)・最低F1**を出し、
「平均が高い」だけでなく「**転んでも大崩れしない**（最低F1が高い）」モデルを評価します。


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(5, shuffle=True, random_state=42)
stability = []
for name, estimator in models.items():
    scores = cross_val_score(estimator, df[features], df["active"], cv=cv, scoring="f1")
    stability.append({"モデル": name, "F1平均": scores.mean(), "F1標準偏差": scores.std(), "最低F1": scores.min()})
pd.DataFrame(stability).sort_values("F1平均", ascending=False).round(3)


### 出力の読み方

- **F1平均**で総合力、**F1標準偏差**で安定度、**最低F1**で最悪ケースを見ます。
- 平均が僅差なら、**標準偏差が小さい方**が実務では安心。平均1位でも最低F1が極端に低いモデルは、条件次第で大外しする危険があります。
- 1回の分割（前セル）と順位が入れ替わることもあります。だから**単発の勝敗で決めない**。これがこの回の教訓です。


## 5人の担当

Dummy / Logistic / Tree / Random Forest / Gradient Boosting を1人ずつ担当し、
**スコア・学習時間・説明しやすさ・安定性**を1行で共有します。「どれが最強か」ではなく
「**この用途にはどれが妥当か**」を言葉にすることが到達目標です。


## 発展（任意）：性能差とモデルを組み合わせる効果を確認する

上位2モデルのF1差が0.01だったとして、それは本物の差でしょうか、それとも分割運でしょうか。
ここでは**反復交差検証＋統計的検定**で差の確からしさを測り、次に複数モデルを組み合わせたときの効果を確認します。


### 反復CV＋Wilcoxon検定：差は偶然でないか

分割の乱数を変えて交差検証を何度も繰り返し（反復CV）、上位2モデルのスコア列を**対応のある検定
（Wilcoxon）**で比べます。p値が小さいほど「差は偶然では説明しにくい」と読めます。


In [ ]:
import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from scipy.stats import wilcoxon

rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=42)
dist = {name: cross_val_score(est, df[features], df["active"], cv=rcv, scoring="f1") for name, est in models.items()}
summary = pd.DataFrame({name: {"F1平均": s.mean(), "F1_SD": s.std()} for name, s in dist.items()}).T.sort_values("F1平均", ascending=False)
display(summary.round(3))
top2 = summary.index[:2].tolist()
stat, p = wilcoxon(dist[top2[0]], dist[top2[1]])
print(f"{top2[0]} vs {top2[1]} のWilcoxon検定 p={p:.3f}（小さいほど差が偶然でない）")


### 出力の読み方

- **p値が0.05より大きい**なら、上位2モデルの差は「偶然の範囲」かもしれず、**わざわざ複雑な方を選ぶ理由は弱い**。
- p値が小さくても、差の**大きさ**（実務的な意味があるか）は別問題。「統計的に有意」と「実務的に重要」は違う、という感覚を持ちます。
- 補足：反復交差検証のスコアは同じデータを使い回すため完全には独立でなく、素朴な検定のp値は**楽観的（有意に出やすい）**になりがちです。ここでは大まかな目安として読み、断定の根拠には使いません。


### 投票・スタッキングで組み合わせる

間違え方の違うモデルを組み合わせると、単体より安定することがあります。**Voting**は予測確率の平均、
**Stacking**は各モデルの予測を入力に上位モデルで統合します。単体最良と比べます。


In [ ]:
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression

estimators = [(name, est) for name, est in models.items() if name != "Dummy"]
ensembles = {
    "Voting(soft)": VotingClassifier(estimators, voting="soft"),
    "Stacking": StackingClassifier(estimators, final_estimator=LogisticRegression(max_iter=1000), cv=5),
}
for name, est in ensembles.items():
    scores = cross_val_score(est, df[features], df["active"], cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="f1")
    print(f"{name:14s} F1平均={scores.mean():.3f} ± {scores.std():.3f}")
print("最良単体:", summary.index[0], "F1平均=", round(summary.iloc[0, 0], 3))


### 出力の読み方

組み合わせたモデルが最良単体を**明確に上回るとは限りません**。効果が出やすいのは、元のモデルたちが
「互いに違う間違え方」をするとき。差がわずかなら、運用の手間を考えて単体を選ぶのも正解です。


### 任意：勾配ブースティング専用ライブラリ

XGBoostが入っていれば試します（`uv sync --extra advanced`）。無い環境では自動でメッセージを出して
スキップし、sklearnの`HistGradientBoosting`で代用できます。


In [ ]:
try:
    from xgboost import XGBClassifier
    xgb = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1, random_state=42, eval_metric="logloss")
    scores = cross_val_score(xgb, df[features].fillna(df[features].median()), df["active"], cv=5, scoring="f1")
    print("XGBoost F1平均:", round(scores.mean(), 3))
except ImportError:
    print("XGBoostは任意です（uv sync --extra advanced）。HistGradientBoostingで代用できます。")


### 出力の読み方

XGBoostのF1が、既に見たGradient Boostingと**近い値**になるはずです。「専用ライブラリ＝必ず勝つ」では
ありません。ライブラリの新しさより、**フェアな比較の枠組み**の方がずっと大事だと、あらためて分かります。


## 追加演習（任意）

モデル比較をさらに多面的に。90分の外の自習向けです。まず2モデルの**学習曲線**を並べ、
「データ追加が効くタイプか」を比べます。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, name in zip(axes, ["Logistic", "Random Forest"]):
    sizes, tr, va = learning_curve(models[name], df[features], df["active"], cv=5, scoring="f1", train_sizes=np.linspace(0.2, 1.0, 5))
    ax.plot(sizes, tr.mean(1), "o-", label="学習")
    ax.plot(sizes, va.mean(1), "o-", label="検証")
    ax.set_title(name); ax.set_xlabel("学習件数"); ax.set_ylabel("F1"); ax.legend()
plt.tight_layout()


### 出力の読み方

- 線形モデル（Logistic）は学習と検証の差が小さい（過学習しにくい）が頭打ちも早い傾向。
- Random Forestは差が大きい（表現力が高く過学習寄り）が、データを増やすと伸びる余地があることも。
- **モデルによってデータ追加の効き方が違う**。増やすか、特徴量を工夫するかの判断材料になります。


### 複数指標を一度に比べる

F1だけでなく、precision・recall・ROC-AUCも同時に交差検証で出します。用途によって重視する指標が
違う（第3回パート2）ので、多面的に見て選びます。


In [ ]:
from sklearn.model_selection import cross_validate, StratifiedKFold

cv = StratifiedKFold(5, shuffle=True, random_state=42)
scoring = ["f1", "precision", "recall", "roc_auc"]
rows = []
for name, est in models.items():
    if name == "Dummy":
        continue
    res = cross_validate(est, df[features], df["active"], cv=cv, scoring=scoring)
    rows.append({"モデル": name, **{m: res[f"test_{m}"].mean() for m in scoring}})
pd.DataFrame(rows).round(3)


### 出力の読み方

あるモデルはrecallが高くprecisionが低い、別のモデルは逆、ということが起きます。**単一のF1では隠れる
個性**が見えます。「見逃しを避けたい」ならrecall列、「空振りを避けたい」ならprecision列で選びます。


### 性能とコストの釣り合い（木の本数）

木の本数（`n_estimators`）を増やすと精度は上がりやすい一方、学習時間も延びます。どこで頭打ちになるかを
見て、**費用対効果**で選びます。


In [ ]:
import time
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score

rows = []
for n in [50, 100, 200, 400]:
    est = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=n, max_depth=6, random_state=42))
    t = time.perf_counter(); est.fit(X_train, y_train); sec = time.perf_counter() - t
    rows.append({"n_estimators": n, "学習秒": sec, "検証F1": f1_score(y_valid, est.predict(X_valid))})
pd.DataFrame(rows).round({"学習秒": 4, "検証F1": 3})


### 出力の読み方

F1はある本数で頭打ちになり、その先は時間だけ延びるはず。**「もう増やしても得しない」点**を見つけるのが
チューニングの勘所。本番のデータ量ではこの差が大きくなるので、小さいうちに感覚をつかんでおきます。


---

# パート2：化学の知識を特徴量にする

**このパートの問い：研究者の知識を、モデルへ渡せる形にするにはどうするか。**


## 特徴量設計＝あなたの化学知識をモデルへ渡す

モデルは与えられた列しか見ません。**「最適温度から離れるほど収率が落ちる」**という知識を持っていても、
`temperature_c`の生の値だけでは、モデルがその山型を学ぶのは大変です。そこで、知識を**計算式**にして
新しい列（特徴量）として渡します。これが特徴量設計です。

鉄則が2つあります。
1. **予測時点で計算できること**（第2回パート2。実験後の値から作らない）。
2. **追加の効果は、同じ検証条件で前後比較して確かめる**（思い込みで良し悪しを決めない）。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


## 演習：仮説を計算式にする

2つの仮説を式にします。**「最適温度78℃からの距離」**（離れるほど収率減、という山型を直接表す）と、
**「単位時間あたりの濃度」**（濃度と時間の兼ね合い）。どちらも計画時に計算できる値です。


In [ ]:
engineered = df.copy()
engineered["temperature_distance"] = (engineered["temperature_c"] - 78).abs()
engineered["concentration_per_hour"] = engineered["concentration_m"] / engineered["reaction_time_h"]
engineered[["temperature_c", "temperature_distance", "concentration_per_hour"]].head()


### 読みどころ

`temperature_distance`は、78℃から上下どちらに離れても大きくなる値（絶対値）。第2回パート1で見た「温度と収率の
山型」を、モデルにとって学びやすい**単調な形**に翻訳しています。生の温度より効くかどうかは、次で検証します。


## アブレーション：追加の効果を「同じ条件」で確かめる

**アブレーション**とは、要素を足し引きして寄与を測る比較のこと。特徴量を追加する前後で、
**同じモデル・同じ交差検証**でMAEを比べます。これをやらずに「良さそうだから採用」は禁物です。


In [ ]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

base = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
added = [*base, "temperature_distance", "concentration_per_hour"]
cv = KFold(5, shuffle=True, random_state=42)
for label, cols in {"追加前": base, "追加後": added}.items():
    est = make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42))
    scores = cross_val_score(est, engineered[cols], engineered["yield_pct"], cv=cv, scoring="neg_mean_absolute_error")
    print(f"{label}: MAE={-scores.mean():.3f} ± {scores.std():.3f}")


### 出力の読み方

- 「追加後」のMAEが「追加前」より**下がっていれば**、その特徴量は効いています。**ばらつき(±)より大きく**下がっているかも見ます（±の中の差は誤差かも）。
- 効かない・悪化することもあります。それも立派な結果。**「この仮説はこのモデルには効かなかった」**と分かるのが検証の価値です。悪化した実験も記録します（第4回パート3）。


## 補足：関連の強い特徴量を選ぶ（相互情報量）

特徴量が増えると、効かない列がノイズになることも。**相互情報量（第2回パート1）**で目的変数との関連が強い順に
並べ、上位k個を選びます。相関と違い、山型のような非線形の関連も拾えます。


In [ ]:
from functools import partial
from sklearn.feature_selection import SelectKBest, mutual_info_regression

# random_stateを固定しないとMIの推定値は実行ごとに変わる（第2回パート1と同じ作法）
mi_score = partial(mutual_info_regression, random_state=42)
sel_data = engineered[added].fillna(engineered[added].median())
selector = SelectKBest(mi_score, k=4).fit(sel_data, engineered["yield_pct"])
pd.DataFrame({"特徴量": added, "MIスコア": selector.scores_, "選択": selector.get_support()}).sort_values("MIスコア", ascending=False).round(3)


### 出力の読み方（結果は素直に受け止める）

MIスコアの高い順に並び、上位4つに「選択=True」が付きます。ここで大事なのは、**自作の`temperature_distance`が
必ず上位に来るとは限らない**ことです。実際、このデータの単変量MIでは上位に来ないことがあります。相互情報量は
**1列ずつ単独で**目的変数との関連を測るため、「他の列と組み合わせて効く」種類の特徴量を低く見積もることがあります。
思い込みで良し悪しを決めず、数字を見る。そして次の発展（任意）/追加演習で、**別の見方（並べ替え重要度）だと結論が
変わる**ことを実際に確かめます。`random_state`を固定しているのは、固定しないとMIの推定値が毎回変わるためです。


## 自由課題（任意）：RDKitでSMILESから記述子を計算する

分子量やLogPは、本来は分子構造（SMILES）から計算できます。RDKitが入っていれば、エタノールの
記述子を実際に計算してみます。無い環境では自動でスキップし、計算済みの列で本編を進められます。


In [ ]:
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Crippen
    molecule = Chem.MolFromSmiles("CCO")
    print("エタノールの分子量:", round(Descriptors.MolWt(molecule), 2))
    print("エタノールのLogP:", round(Crippen.MolLogP(molecule), 2))
except ImportError:
    print("RDKitは任意（uv sync --extra chemistry）。計算済みmolecular_weight/logp/tpsaで本編を進められます。")


### 読みどころ

RDKitが動けば、SMILES（`CCO`＝エタノール）から分子量やLogPが再現されます。**「記述子＝構造から計算できる
特徴量」**だと腹落ちします。RDKitは発展扱いなので、無くても計算済みの列で全く問題ありません。


## 発展（任意）：リークしやすい特徴量と、賢い選択

強力だが**リークしやすい**特徴量の代表が**target encoding**（カテゴリを目的変数の平均で置き換える）です。
やり方を誤ると、第2回パート3で学んだリークを自ら仕込むことになります。安全なやり方を身につけます。


### target encoding：全データ平均は「リーク」、OOFなら安全

「系列ごとの平均収率」を特徴量にしたいとします。**全データの平均**で作ると、各行の答えが自分の特徴量に
混ざりリークします。正しくは、第2回パート3の交差検証と同じ発想で、**その行を含まない分割の平均**で作ります
（OOF＝out-of-fold）。両者でMAEを比べ、リークが楽観を生むことを確かめます。


In [ ]:
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Ridge

def oof_target_encode(frame, col, target, n_splits=5, seed=42):
    "分割の内側で平均を学習するリーク安全なtarget encoding。"
    encoded = pd.Series(index=frame.index, dtype=float)
    global_mean = frame[target].mean()
    for tr, va in KFold(n_splits, shuffle=True, random_state=seed).split(frame):
        means = frame.iloc[tr].groupby(col)[target].mean()
        encoded.iloc[va] = frame.iloc[va][col].map(means).fillna(global_mean).to_numpy()
    return encoded

leaky = df["scaffold_group"].map(df.groupby("scaffold_group")["yield_pct"].mean())
safe = oof_target_encode(df, "scaffold_group", "yield_pct")
num_cols = ["temperature_c", "concentration_m", "logp"]
X_num = df[num_cols].fillna(df[num_cols].median())
for label, enc in {"リークあり(全データ平均)": leaky, "OOF(安全)": safe}.items():
    feats = X_num.assign(scaffold_te=enc.to_numpy())
    scores = cross_val_score(Ridge(), feats, df["yield_pct"], cv=5, scoring="neg_mean_absolute_error")
    print(f"{label}: MAE={-scores.mean():.3f}")
print("リークありは楽観的に見えることがある。実運用の性能はOOFに近い。")


### 出力の読み方

「リークあり」のMAEが「OOF」より**小さく（良く）見える**ことがあります。しかしそれは幻。本番では
その行の答えは手に入りません。**実運用の実力はOFFの側**。強力な特徴量ほど、作り方のリークに注意します。


### RFECV：交差検証つきで特徴量を絞り込む

`RFECV`は、重要度の低い特徴量を1つずつ削りながら交差検証し、**性能が最も良くなる特徴量の組**を
自動で選びます。人手の取捨選択より客観的です。ここでは**係数の大きさで重要度を測る線形モデル(Ridge)**で
回します（後述のとおり、木モデルはノイズに強すぎてRFECVが列を削らないことが多いため）。尺度をそろえてから
かけます。


In [ ]:
import numpy as np
from sklearn.feature_selection import RFECV
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# わざと「無意味な列」を混ぜて、RFECVがそれを削れるかを確かめる
rng = np.random.default_rng(0)
rfe_data = engineered[added].fillna(engineered[added].median()).copy()
rfe_data["noise"] = rng.normal(size=len(rfe_data))     # 目的変数と無関係な乱数列
rfe_data["logp_copy"] = rfe_data["logp"]                # 既存列の複製（冗長）
scaled = pd.DataFrame(StandardScaler().fit_transform(rfe_data), columns=rfe_data.columns, index=rfe_data.index)
rfecv = RFECV(Ridge(alpha=1.0), cv=5, scoring="neg_mean_absolute_error", min_features_to_select=2)
rfecv.fit(scaled, engineered["yield_pct"])
print("元の列数:", scaled.shape[1], "→ 選ばれた列数:", rfecv.n_features_)
pd.DataFrame({"特徴量": rfe_data.columns, "残す": rfecv.support_, "順位": rfecv.ranking_}).sort_values("順位")


### 出力の読み方

- `残す=True`が採用列、`順位=1`が最重要グループ。**わざと混ぜた`noise`（乱数）と`logp_copy`（複製）が削られていれば**、RFECVが「役に立たない列を見抜いて外す」働きをしていると確認できます。
- 木モデル(RandomForest)ではなく線形モデル(Ridge)を使ったのは、**木モデルはノイズ列があっても性能が落ちにくく、RFECVが何も削らないことが多い**ため。**推定器を変えると選択結果も変わる**。特徴量選択も「どの手法で測るか」に依存する、という点も併せて押さえます。
- 選択も交差検証の内側で行うことで、選びすぎ（過学習）を避けています。


## 追加演習（任意）

特徴量づくりの引き出しを増やします。90分の外の自習向けです。まず**交互作用特徴量**。2つの列の
掛け算で「組み合わせの効果」を表します（温度×濃度など）。


In [ ]:
from sklearn.preprocessing import PolynomialFeatures

cols = ["temperature_c", "concentration_m"]
pair = engineered[cols].fillna(engineered[cols].median())
inter = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
out = inter.fit_transform(pair)
display(pd.DataFrame(out, columns=inter.get_feature_names_out(), index=pair.index).head())


### 出力の読み方

元の2列に加え、`temperature_c concentration_m`（掛け算）の列ができます。`interaction_only=True`なので
二乗は作らず組み合わせだけ。「片方が高いときだけもう片方が効く」ような関係を、モデルへ渡せます。


### 連続値を区間に区切る（ビニング）

温度のような連続値を4区間に区切ると、非線形な効果を扱いやすくなったり、解釈しやすくなったりします。
`KBinsDiscretizer`（分位点で等件数に区切る）を使い、区間ごとの平均収率を見ます。


In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

temp = engineered[["temperature_c"]].fillna(engineered["temperature_c"].median())
binner = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="quantile")
engineered["temp_bin"] = binner.fit_transform(temp).astype(int)
display(engineered.groupby("temp_bin")["yield_pct"].mean().round(1))


### 出力の読み方

区間0（低温）〜3（高温）ごとの平均収率が出ます。中間の区間で収率が高い（山型）なら、第2回パート1で見た
温度の効果と一致。ビニングは効果を見せやすい一方、情報を捨てる面もあるので、元の連続値と併用も検討します。


### 作った特徴量の効き目を、並べ替え重要度で確かめる

第1・12回で使った並べ替え重要度を、この回で作った特徴量を含めた全体に適用します。自作特徴量が
上位に来るかを、holdoutで公平に確認します。


In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

Xe = engineered[added].fillna(engineered[added].median())
Xtr, Xva, ytr, yva = train_test_split(Xe, engineered["yield_pct"], test_size=0.25, random_state=42)
rf = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42).fit(Xtr, ytr)
perm = permutation_importance(rf, Xva, yva, scoring="neg_mean_absolute_error", n_repeats=15, random_state=42)
pd.DataFrame({"特徴量": added, "重要度": perm.importances_mean}).sort_values("重要度", ascending=False).round(3)


### 出力の読み方：3つの見方が食い違うのは正常

ここでは`temperature_distance`が**上位に来ることがあります**。ところが同じ回の基本では、相互情報量(MI)で
同じ列が**下位**、アブレーションでは追加しても**MAEがほとんど改善しない**。3つの見方で結論が食い違います。
矛盾ではなく、**それぞれ別の問いに答えているから**です。

- **アブレーション**：その列を入れるか抜くかで最終性能がどう動くか。他の列で代用が効くと、抜いても悪化せず「効果なし」に見える。
- **相互情報量**：その列を単独で見たときの関連の強さ。組み合わせて効く効果は測れない。
- **並べ替え重要度**：学習済みモデルが実際にその列に依存しているか。`temperature_distance`は`temperature_c`から作った相関の強い列なので、モデルがどちらを使うかで重要度が振れやすい。

教訓は2つ。**(1) 1つの指標だけで特徴量の良し悪しを断じない。(2) 元の列と強く相関する派生列（今回の距離特徴量）は、
重要度が不安定になりやすい。** 「作る→交差検証で効果を確かめる→複数の見方で解釈する」という一巡こそが、
思い込みを避ける特徴量設計です。


---

# パート3：改善実験を1つずつ行う

**このパートの問い：改善した理由を後から説明できる実験とは何か。**


## 「なんとなく良くなった」を卒業する

改善は勢いでやると、後で「なぜ良くなったのか」を説明できません。この回のテーマは、**理由を後から
説明できる実験のやり方**です。次の原則が効きます。

1. **一度に変えるのは1つだけ**（複数変えると、どれが効いたか分からない）。
2. **比較条件は固定**（同じ分割・同じ指標）。
3. **結果は平均とばらつきで残す**（1回のスコアで一喜一憂しない）。
4. **良くなった実験も悪くなった実験も記録する**（消さない）。

まず、すべての実験で共通して使うデータと交差検証を用意します。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X, y = df[features], df["active"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


## 演習：1要素だけ変えて、実験ログに残す

`max_depth`**だけ**を変えた3つの実験を回し、結果を表（実験ログ）にします。他の設定は固定。
学習F1と検証F1平均を両方残すのは、**過学習の度合い**（第2回パート3）も一緒に記録するためです。


In [ ]:
rows = []
for depth in [3, 6, None]:
    model = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=150, max_depth=depth, random_state=42))
    scores = cross_validate(model, X, y, cv=cv, scoring="f1", return_train_score=True)
    rows.append({"実験名": f"depth={depth}", "変更点": "max_depthのみ",
                 "学習F1": scores["train_score"].mean(), "検証F1平均": scores["test_score"].mean(),
                 "検証F1標準偏差": scores["test_score"].std()})
experiment_log = pd.DataFrame(rows)
experiment_log.round(3)


### 出力の読み方

- **検証F1平均が最も高い深さ**が候補。ただし**検証F1標準偏差**が大きいなら、その優位は不安定かもしれません。
- 差が標準偏差より小さいなら「実質同じ」と読み、より単純な（浅い）設定を選ぶのが無難です。
- `depth=None`で学習F1が跳ね上がり検証F1が伸びないなら、過学習。**表1つで「効果」と「過学習」を同時に管理**できます。


## 実験ログの最小項目

同期回でも自習でも、次を1行で残せば十分です。

- **実験名 / 変えたもの（1つ）/ 固定した比較条件 / 結果の平均とばらつき / 分かったこと / 次の仮説**

Copilotには次の実験案を出してもらってもよいですが、**優先順位と「予測時点で妥当か」の判断は人**が行います。


## 発展（任意）：探索を自動化し、正直な推定を得る

手で`max_depth`を変えるのは学習には良いですが、設定が増えると大変です。**探索の自動化**と、
第2回パート3で学んだ**ネストCV（正直な推定）**、そして**重要度を区間で読む**ことを扱います。


### RandomizedSearchCV：設定を自動で探す

複数の設定候補から無作為に組み合わせを試し、交差検証で最良を選びます。総当たり（GridSearch）より
少ない回数で広く探せるのが利点。`n_iter`が試行回数です。


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

pipe = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(random_state=42))
param_dist = {
    "randomforestclassifier__n_estimators": [100, 200, 300],
    "randomforestclassifier__max_depth": [3, 4, 6, None],
    "randomforestclassifier__min_samples_leaf": [1, 2, 4],
    "randomforestclassifier__max_features": ["sqrt", "log2", None],
}
search = RandomizedSearchCV(pipe, param_dist, n_iter=10, cv=cv, scoring="f1", random_state=42)
search.fit(X, y)
print("最良設定:", search.best_params_)
print("探索内での最良CV F1:", round(search.best_score_, 3))


### 出力の読み方と、大事な注意

`best_params_`が選ばれた設定、`best_score_`がそのCF1です。**ただしこの`best_score_`をそのまま「性能」として
報告してはいけません**。たくさん試して一番良かった数字なので、下駄を履いています。次で正直な推定に直します。


### ネストCV：探索の下駄を脱いだ推定

「探索」を1つのモデルとみなし、その外側でもう一段の交差検証をかけます。各外側分割で設定を選び直し、
未見のデータで評価するので、**探索による楽観が乗らない正直な性能**が得られます。


In [ ]:
from sklearn.model_selection import cross_val_score

outer = StratifiedKFold(5, shuffle=True, random_state=7)
nested = cross_val_score(search, X, y, cv=outer, scoring="f1")
print("ネストCV外側F1:", nested.round(3))
print("楽観の少ない推定:", round(nested.mean(), 3), "±", round(nested.std(), 3), " ← 探索内スコアより低いのが普通")


### 出力の読み方

ネストCVの平均は、前セルの`best_score_`より**少し低い**のが普通で、その差が「探索による楽観」の大きさです。
論文や報告に載せるなら、こちらの正直な数字を使います。


### 並べ替え重要度は「区間」で読む

第1回パート1で見た並べ替え重要度を、今度は**ばらつき（±2SD）つき**で読みます。下限が0を跨ぐ特徴量は
「効いているとは言い切れない」。評価は学習に使っていない**holdout**で行い、公平性を保ちます。


In [ ]:
from sklearn.model_selection import train_test_split

X_fit, X_holdout, y_fit, y_holdout = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
best = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)).fit(X_fit, y_fit)
perm = permutation_importance(best, X_holdout, y_holdout, scoring="f1", n_repeats=30, random_state=42)
importance = pd.DataFrame({
    "特徴量": features,
    "重要度平均": perm.importances_mean,
    "下限(平均-2SD)": perm.importances_mean - 2 * perm.importances_std,
}).sort_values("重要度平均", ascending=False)
importance["0を跨ぐ"] = importance["下限(平均-2SD)"] <= 0
importance.round(4)


### 出力の読み方

- `0を跨ぐ=True`の特徴量は、**寄与があるとは断言できない**（ばらつきの範囲に0が入る）。
- 上位で`0を跨ぐ=False`の特徴量が、自信を持って「効いている」と言える列。ここから**反証可能な次の仮説**
（「この列を強める特徴量を足したら改善するのでは？」）を1つ立てて、基本の実験ログへ戻ります。これが改善実験です。


## 追加演習（任意）

実験の回し方を仕組み化します。90分の外の自習向けです。まず**実験を1行で記録する関数**を作り、
複数の設定を回してログに溜めます。手作業のコピペより、記録漏れが減ります。


In [ ]:
from sklearn.model_selection import cross_val_score

experiment_log = []
def run_experiment(name, estimator, note=""):
    "設定を交差検証で評価し、実験ログへ1行追加して返す。"
    scores = cross_val_score(estimator, X, y, cv=cv, scoring="f1")
    row = {"実験名": name, "F1平均": round(scores.mean(), 3), "F1_SD": round(scores.std(), 3), "分かったこと": note}
    experiment_log.append(row)
    return row

run_experiment("depth3", make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=150, max_depth=3, random_state=42)), "浅め")
run_experiment("depth6", make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42)), "標準")
run_experiment("leaf4", make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=150, max_depth=6, min_samples_leaf=4, random_state=42)), "葉を大きく")
pd.DataFrame(experiment_log)


### 出力の読み方

3つの実験がログにたまり、F1平均・ばらつき・分かったことが1表に。**変更点と結果がセットで残る**ので、後から
「なぜこの設定にしたか」を説明できます。関数化しておくと、実験のたびに1行呼ぶだけで済みます。


### 検証曲線：1つの設定を動かして最適点を探す

`validation_curve`は、1つのハイパーパラメータ（ここでは`max_depth`）を動かし、学習と検証のスコア推移を
描きます。最適な複雑さが視覚的に分かります。


In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import validation_curve

depths = [2, 3, 4, 6, 8, 12]
tr, va = validation_curve(
    make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=150, random_state=42)),
    X, y, param_name="randomforestclassifier__max_depth", param_range=depths, cv=cv, scoring="f1",
)
plt.plot(depths, tr.mean(1), "o-", label="学習")
plt.plot(depths, va.mean(1), "o-", label="検証")
plt.xlabel("max_depth"); plt.ylabel("F1"); plt.legend(); plt.title("検証曲線")
plt.tight_layout()


### 出力の読み方

学習F1は深さとともに上がり続けますが、検証F1は途中で頭打ち・下降します。**検証F1が最大になる手前**が
最適な深さ。2本の乖離が広がるほど過学習が進んでいる、という第2回パート3の読み方がそのまま使えます。


### 実験ログをファイルに残す

ログをCSVに保存し、読み直します。セッションをまたいで実験を積み上げられ、再現性（第5回パート3）にもつながります。


In [ ]:
out = ROOT / "workspace" / "experiment_log.csv"
pd.DataFrame(experiment_log).to_csv(out, index=False)
reloaded = pd.read_csv(out)
print("保存＆再読込した実験ログ:", out)
display(reloaded)


### 出力の読み方

`workspace/experiment_log.csv`に保存され、読み直しても同じ内容。**記録を残す文化**が、思いつきの改善を
再現可能な知見へ変えます。良い変更も悪い変更も、まずログに残すことを、この回でいちばんの習慣にしてください。


---

## よくある誤り

- 異なる分割で比較する
- モデルごとに異なる指標を報告する
- 最も高い1回のスコアだけを採用する
- 意味を説明できない特徴量を大量追加する
- 目的変数由来の値を全データで作って特徴量にする
- 追加前後で分割やモデルも変える
- 同時に複数要素を変える
- 探索に使った分割で最終性能も報告する
- 悪化した実験を記録から消す

## 自習（任意・30〜60分）

- RepeatedStratifiedKFoldでF1分布を作り、上位2モデルをWilcoxon検定で比べる
- StackingClassifierと最良単体のF1・学習時間を比較する
- 自作KFold target encodingの有無でMAEを比較する
- RFECVで残った特徴量と、化学的な解釈を突き合わせる
- RandomizedSearchCVの最良設定を、ネストCVの外側スコアで確かめる
- 並べ替え重要度を20反復で計算し、区間が0を跨ぐ列を挙げる

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. 公平な比較に固定すべきものは何か
2. 対応のある検定が必要な理由は何か
3. スタッキングが効きやすいのはどんなときか
4. その特徴量はいつ計算できるか
5. target encodingでリークを防ぐ手順は何か
6. 特徴量選択も交差検証の内側で行う理由は何か
7. 1要素だけ変える理由は何か
8. ネストCVは何を防ぐか
9. 重要度の区間が0を跨ぐとどう解釈するか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
